# 02a — Arabic Cleaning & Normalization

In [ ]:
import re
import html as html_module
import json
from pathlib import Path

RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Arabic diacritics (tashkeel) range
ARABIC_DIACRITICS = re.compile(r"[ً-ْٰـ]")

ALEF_VARIANTS = re.compile(r"[إأآا]")
YEH_VARIANTS = re.compile(r"[يى]")
TEH_MARBUTA = re.compile(r"ة")

STYLE_BLOCK = re.compile(r"<style[^>]*>.*?</style>", re.DOTALL | re.IGNORECASE)
HTML_TAG = re.compile(r"<[^>]+>")


def strip_html(text: str) -> str:
    text = STYLE_BLOCK.sub(" ", text)
    text = HTML_TAG.sub(" ", text)
    text = html_module.unescape(text)
    return text


def normalize_arabic(text: str, strip_diacritics: bool = True) -> str:
    text = strip_html(text)
    if strip_diacritics:
        text = ARABIC_DIACRITICS.sub("", text)
    text = ALEF_VARIANTS.sub("ا", text)
    text = YEH_VARIANTS.sub("ي", text)
    # teh marbuta normalization intentionally not applied by default

    text = re.sub(r"\s+", " ", text).strip()
    return text


def preserve_citation_markers(text: str) -> str:
    return text


### Load raw scraped records from each source and normalize

In [2]:
def load_and_normalize(source_name: str, filenames, text_field: str = "text"):
    if isinstance(filenames, str):
        filenames = [filenames]
    records = []
    for filename in filenames:
        path = RAW_DIR / source_name / filename
        if not path.exists():
            print(f"SKIP {path} — not scraped yet")
            continue
        file_records = json.loads(path.read_text(encoding="utf-8"))
        records.extend(file_records)

    for r in records:
        raw = r.get(text_field) or r.get("html", "")
        r["normalized_text"] = normalize_arabic(raw)

    out_path = OUT_DIR / f"{source_name}_normalized.json"
    out_path.write_text(json.dumps(records, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{source_name}: normalized {len(records)} records -> {out_path}")
    return records


SJC_TYPE_FILES = [
    "sjc_مدني.json", "sjc_جنائي.json", "sjc_شرعي.json",
    "sjc_تجاري.json", "sjc_انتخابات.json", "sjc_توحيد_المبادئ.json",
]

sjc_records = load_and_normalize("sjc", SJC_TYPE_FILES)
ccb_records = load_and_normalize("ccb", "ccb_rulings.json")
lloc_records = load_and_normalize("lloc", "lloc_legislation.json")


sjc: normalized 9094 records -> ..\data\processed\sjc_normalized.json


ccb: normalized 93 records -> ..\data\processed\ccb_normalized.json


lloc: normalized 588 records -> ..\data\processed\lloc_normalized.json
